# MedStream - Data Warehouse Ingestion (DuckDB)

Stores processed patient records into an embedded DuckDB analytical database (`medstream_warehouse.duckdb`) and runs verification queries.

### 1. Setup & Connection

In [ ]:
import duckdb
import pandas as pd

db_file = "medstream_warehouse.duckdb"
con = duckdb.connect(db_file)
print(f"Connected to warehouse: {db_file}")

### 2. Load Cleaned Data into Warehouse Table

In [ ]:
cleaned_csv = "data/processed/diabetes_cleaned.csv"
df_clean = pd.read_csv(cleaned_csv)

# register and persist table
con.register('df_source', df_clean)
con.execute("CREATE OR REPLACE TABLE patients_analytics AS SELECT * FROM df_source;")
print("Persisted 'patients_analytics' table")

### 3. Verify Record Count

In [ ]:
total = con.execute("SELECT COUNT(*) FROM patients_analytics").fetchone()[0]
print(f"Total rows in table: {total:,}")

### 4. Analytical Aggregation Query

In [ ]:
query = """
SELECT 
    CASE WHEN diabetes = 1 THEN 'Diabetic' ELSE 'Non-Diabetic' END AS diagnosis,
    COUNT(*) AS patient_count,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(AVG(blood_glucose_level), 1) AS mean_glucose,
    ROUND(AVG(HbA1c_level), 2) AS mean_hba1c,
    ROUND(AVG(bmi), 1) AS mean_bmi
FROM patients_analytics
GROUP BY diabetes;
"""

summary = con.execute(query).df()
summary

### 5. Close Connection

In [ ]:
con.close()
print("Warehouse connection closed.")